In [3]:
pip list

Package                           Version
--------------------------------- -------------------
aiobotocore                       2.19.0
aiohappyeyeballs                  2.4.4
aiohttp                           3.11.10
aioitertools                      0.7.1
aiosignal                         1.2.0
alabaster                         0.7.16
altair                            5.5.0
anaconda-anon-usage               0.7.1
anaconda-auth                     0.8.6
anaconda-catalogs                 0.2.0
anaconda-cli-base                 0.5.2
anaconda-client                   1.13.0
anaconda-navigator                2.6.6
anaconda-project                  0.11.1
annotated-types                   0.6.0
anyio                             4.7.0
appdirs                           1.4.4
archspec                          0.2.3
argon2-cffi                       21.3.0
argon2-cffi-bindings              21.2.0
arrow                             1.3.0
astroid                           3.3.8
astropy         

In [5]:
import pandas as pd
data=pd.read_csv("telecom_data.csv")
#data['timestamps'] = pd.to_datetime(data['timestamps'])
print(data.head(10))
data.isnull().sum()


   Age  Gender  PlanType  MonthlyUsage Churn
0   21  Female   Regular            15    No
1   45  Female   Economy            41    No
2   44  Female   Economy            40    No
3   31  Female   Regular            23   Yes
4   33  Female   Regular            12    No
5   42  Female   Regular            52    No
6   20  Female     Ultra            57   Yes
7   26    Male     Ultra            23    No
8   37  Female  Advanced            31    No
9   26    Male   Economy            23    No


Age             0
Gender          0
PlanType        0
MonthlyUsage    0
Churn           0
dtype: int64

In [3]:
print('Columns:', data.columns.tolist())
print('\nDataset Info:')
print(data.info())
print('\nDataset Completeness:')
print(data.isnull().sum())
print('\nDataset Consistency: ')
print(data.dtypes)

Columns: ['Age', 'Gender', 'PlanType', 'MonthlyUsage', 'Churn']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Age           150 non-null    int64 
 1   Gender        150 non-null    object
 2   PlanType      150 non-null    object
 3   MonthlyUsage  150 non-null    int64 
 4   Churn         150 non-null    object
dtypes: int64(2), object(3)
memory usage: 6.0+ KB
None

Dataset Completeness:
Age             0
Gender          0
PlanType        0
MonthlyUsage    0
Churn           0
dtype: int64

Dataset Consistency: 
Age              int64
Gender          object
PlanType        object
MonthlyUsage     int64
Churn           object
dtype: object


In [7]:
print('\nDataset Describe:')
data.describe()


Dataset Describe:


,Age,MonthlyUsage
count,150.000000,150.000000
mean,35.193333,33.693333
std,10.841566,15.923031
min,19.000000,3.000000
25%,25.000000,23.000000
50%,35.000000,35.000000
75%,44.000000,50.000000
max,54.000000,59.000000


In [9]:
#BIAS
print(data['Gender'].value_counts(normalize=True))
print(data['Churn'].value_counts(normalize=True))

Gender
Female    0.793333
Male      0.206667
Name: proportion, dtype: float64
Churn
No     0.893333
Yes    0.106667
Name: proportion, dtype: float64


In [13]:
fdata = data.drop(columns=['Churn'])
X=fdata
y=data["Churn"].map({'Yes':1, 'No':0})
categorical = fdata.select_dtypes(include='object').columns
numerical = fdata.select_dtypes(exclude='object').columns
print("Categorial Features:", list(categorical))
print("Numerical Features:", list(numerical))

Categorial Features: ['Gender', 'PlanType']
Numerical Features: ['Age', 'MonthlyUsage']


In [25]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
X_encoded_gd=pd.get_dummies(
X,
columns=categorical,
drop_first=True
)
#print(X_encoded_gd.Gender_Male)
#print(X_encoded_gd.PlanType_Regular)
print("gd Encoded Columns: ", X_encoded_gd.columns.tolist())
ohe = OneHotEncoder(
    drop='first',
    sparse_output=False
)

encoded_array = ohe.fit_transform(X[categorical])

encoded_df = pd.DataFrame(
    encoded_array,
    columns=ohe.get_feature_names_out(categorical)
)

X_encoded_ohe = pd.concat(
    [X[numerical].reset_index(drop=True), encoded_df.reset_index(drop=True)],
    axis=1
)
print(encoded_array)
print("OHE Encoded Columns: ",X_encoded_ohe.columns.tolist())

scaler = StandardScaler()
X_scaled_gd = X_encoded_gd.copy()
X_scaled_gd[numerical] = scaler.fit_transform(X_scaled_gd[numerical])
X_scaled_ohe = X_encoded_ohe.copy()
X_scaled_ohe[numerical] = scaler.fit_transform(X_scaled_ohe[numerical])
df= pd.DataFrame({
    'Age_ohe':X_scaled_ohe[numerical]['Age'],
    'Age_gd':X_scaled_gd[numerical]['Age'],
    'MonthlyUsage_ohe':X_scaled_ohe[numerical]['MonthlyUsage'],
    'MonthlyUsage_gd':X_scaled_gd[numerical]['MonthlyUsage']
})
print("Comparison_Table:\n",df)

gd Encoded Columns:  ['Age', 'MonthlyUsage', 'Gender_Male', 'PlanType_Economy', 'PlanType_Regular', 'PlanType_Standard', 'PlanType_Ultra']
[[0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]
 [1. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 1. 0. 0.]
 [1. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0.]
 [1. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0

In [17]:
print(X_encoded_gd.Gender_Male)

0      False
1      False
2      False
3      False
4      False
       ...  
145    False
146    False
147    False
148    False
149    False
Name: Gender_Male, Length: 150, dtype: bool


In [18]:
print(X_encoded_gd.PlanType_Regular)

0       True
1      False
2      False
3       True
4       True
       ...  
145    False
146    False
147    False
148    False
149    False
Name: PlanType_Regular, Length: 150, dtype: bool
